In [1]:
!pip install -q openai

# Environment Serup



In [29]:
from pprint import pprint
from pydantic import BaseModel

def print_response(response):
  print(f"Response id: {response.id}")

  # NOTE: If `:free` models are used, the output tokens may be set to zero.
  # print(f"Input tokens: {response.usage.input_tokens} ({response.usage.input_tokens_details.cached_tokens} cached); Output tokens: {response.usage.output_tokens} ({response.usage.output_tokens_details.reasoning_tokens} reasoning)")
  print(f"Input tokens: {response.usage.input_tokens} ({getattr(response.usage.input_tokens_details, 'cached_tokens', None)} cached); Output tokens: {response.usage.output_tokens} ({getattr(response.usage.output_tokens_details, 'reasoning_tokens', None)} reasoning)" if response.usage is not None else "Usage information was not returned.")
  pprint(response.output)

  print()
  print(f"{'-' * 20} [Text] {'-' * 20}")
  print(response.output_text)

# Basic usage

In [3]:
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get('OPENROUTER_API_KEY')
client = OpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")

In [4]:
available_models = client.models.list()

In [5]:
pprint(available_models.data)

[Model(id='poolside/laguna-xs-2.1:free', created=1783002429, object=None, owned_by=None, canonical_slug='poolside/laguna-xs-2.1-20260625', hugging_face_id='poolside/Laguna-XS-2.1', name='Poolside: Laguna XS 2.1 (free)', description='Laguna XS 2.1 is the latest coding agent model in the 33B-A3B category from [Poolside](https://poolside.ai/) and a step forward from their Laguna XS.2 model (released in April 2026). It combines...', context_length=262144, architecture={'modality': 'text->text', 'input_modalities': ['text'], 'output_modalities': ['text'], 'tokenizer': 'Other', 'instruct_type': None}, pricing={'prompt': '0', 'completion': '0'}, top_provider={'context_length': 262144, 'max_completion_tokens': 32768, 'is_moderated': False}, per_request_limits=None, supported_parameters=['include_reasoning', 'max_tokens', 'reasoning', 'temperature', 'tool_choice', 'tools'], default_parameters={'temperature': 0.7, 'top_p': 0.9, 'top_k': None, 'frequency_penalty': None, 'presence_penalty': None, 

In [7]:
simple_response = client.responses.create(
    model="google/gemma-3-4b-it",
    input="Hello World AI"
)

In [8]:
print_response(simple_response)

Response id: gen-1783084646-5hUwhUPxOmSsdWlJvPEl
Input tokens: 12 (0 cached); Output tokens: 197 (0 reasoning)
[ResponseOutputMessage(id='msg_tmp_4nd14s4vjjj', content=[ResponseOutputText(annotations=[], text="Hello there! It's great to connect with you. I'm a large language model, and you can think of me as an AI assistant. I'm here to help you with a wide variety of tasks, like:\n\n* **Answering your questions:** I can try to provide informative answers, even if they're complex or open-ended.\n* **Generating text:**  I can write stories, poems, scripts, code, articles – you name it!\n* **Summarizing text:**  Give me a long piece of text and I'll condense it for you.\n* **Translating languages:** I can translate between many different languages.\n* **Brainstorming ideas:** Feeling stuck? Let's explore some possibilities together.\n\n**How can I help you today?**  Do you want to:\n\n* Ask me a question?\n* Request a creative writing prompt?\n*  Try a different task altogether?", type='

# Streaming

In [10]:
with client.responses.stream(
    model="google/gemma-3-4b-it",
    input="Hello World AI",
) as stream:
    for event in stream:
        pprint(event)

    streamed_response = stream.get_final_response()

ResponseCreatedEvent(response=Response(id='gen-1783085320-GpI08RRP1Tw0pvDsDzlB', created_at=1783085320.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='google/gemma-3-4b-it', object='response', output=[], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, completed_at=None, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention=None, reasoning=None, safety_identifier=None, service_tier='auto', status='in_progress', text=ResponseTextConfig(format=ResponseFormatText(type='text'), verbosity=None), top_logprobs=0, truncation='disabled', usage=None, user=None, presence_penalty=0, frequency_penalty=0, store=False), sequence_number=0, type='response.created')
ResponseInProgressEvent(response=Response(id='gen-1783085320-GpI08RRP1Tw0pvDsDzlB', created_at=1783085320.0, error=None, incomplete_details=None, 

In [11]:
print_response(streamed_response)

Response id: gen-1783085320-GpI08RRP1Tw0pvDsDzlB
Input tokens: 12 (0 cached); Output tokens: 189 (0 reasoning)
[ParsedResponseOutputMessage[TypeVar](id='msg_tmp_6ck1vjnhk63', content=[ParsedResponseOutputText[TypeVar](annotations=[], text="Hello there! It's nice to meet you. I'm a large language model, a type of AI, and I'm here to chat, answer your questions, and try my best to be helpful. \n\nYou can call me Hello World AI – it’s a fun way to remember who I am! \n\n**What would you like to do or talk about today?** \n\nHere are a few things I can do:\n\n*   **Answer questions:** About almost anything!\n*   **Write stories, poems, or code:** Let me know what you're looking for.\n*   **Translate languages:** I can translate between many languages.\n*   **Summarize text:** Give me a long article, and I'll give you the key points.\n*   **Just chat:** Sometimes it's nice to just have a conversation.\n\n\nLet's start! 😊", type='output_text', logprobs=[], parsed=None)], role='assistant', st

# Reasoning

For this section, pick models that support the `reasoning` parameter.

In [12]:
low_reasoning_response = client.responses.create(
    model="minimax/minimax-m2.5:floor",
    input="Give me top 3 Ideas for IT Startup in 2027",
    reasoning={"effort": "low"}
)

In [16]:
print_response(low_reasoning_response)

Response id: gen-1783085490-9uUtWRFLxxjg2b0nGkVU
Input tokens: 51 (0 cached); Output tokens: 1380 (257 reasoning)
[ResponseReasoningItem(id='rs_tmp_b33v5skzaiq', summary=[], type='reasoning', content=[Content(text='The user asks: "Give me top 3 Ideas for IT Startup in 2027". We have to consider the policy: It\'s a request for business ideas, not disallowed. It\'s allowed. The user wants top 3 ideas for an IT startup in 2027. We can respond with potential startup ideas, perhaps focusing on emerging technologies and trends predicted for 2027, like AI-driven solutions, quantum computing, edge computing, AR/VR, sustainability, healthcare IT, cybersecurity, etc. We must ensure compliance with policy. There\'s no disallowed content. It\'s permissible. We need to respond with three ideas.\n\nWe should note that it\'s "2027", but we can discuss predictions based on current trends and emerging tech: AI agents for business processes, quantum computing cloud services, green tech for data centers,

In [14]:
high_reasoning_response = client.responses.create(
    model="minimax/minimax-m2.5:floor",
    input="Give me top 3 Ideas for IT Startup in 2027",
    reasoning={"effort": "high"}
)

In [15]:
print_response(high_reasoning_response)

Response id: gen-1783085512-g0jUVtebIFQeHB8dP43P
Input tokens: 51 (0 cached); Output tokens: 2097 (461 reasoning)
[ResponseReasoningItem(id='rs_tmp_lcvinggvo7', summary=[], type='reasoning', content=[Content(text='The user asks: "Give me top 3 Ideas for IT Startup in 2027". We need to propose top 3 IT startup ideas likely relevant in 2027. This is a legitimate request. There\'s no disallowed content. We can propose ideas that incorporate emerging trends: AI-driven solutions, sustainability, health tech, quantum computing, edge computing, Web3, AR/VR, autonomous systems, etc. The answer should be well-structured, provide a brief description, market opportunity, and why it\'s promising in 2027. Must be safe and follow policy. Provide three ideas. Possibly include a note about market size, competition, or required expertise. The user likely wants ideas.\n\nWe should keep it short but thorough: each idea with a brief description, market drivers, and potential products. Also note that this 

# **Multi-turn conversations**

This example includes `instructions` giving the model high-level guidance on how it should behave while generating a response.

> **OpenAI:**
> _Note that the `instructions` parameter only applies to the current response generation request. If you are managing conversation state with the `previous_response_id` parameter, the `instructions` used on previous turns will not be present in the context._

By default, OpenAI persists each response on their servers. This is what enables the `previous_response_id` parameter - at the server-side the full context is reconstructed before it hits the model.

> **OpenAI:**
> _Even when using previous_response_id, all previous input tokens for responses in the chain are billed as input tokens in the API. [Conversation state](https://developers.openai.com/api/docs/guides/conversation-state)_

In [17]:
multi_turn_conversation = [{ "role": "user", "content": "Generate a short role-play in no more than 5 lines: manager and employee discussing performance issues." }]
first_turn_response = client.responses.create(
    model="openai/gpt-oss-120b",
    input=multi_turn_conversation,
    instructions="The manager should talk like a pirate.",
    extra_body={
        "provider": { "sort": { "by": "throughput" } }
    }
)

In [18]:
print_response(first_turn_response)

Response id: gen-1783085684-M4yIq1weJPo1qdzaA79C
Input tokens: 102 (0 cached); Output tokens: 244 (119 reasoning)
[ResponseReasoningItem(id='rs_tmp_l1dr2vg3lzk', summary=[], type='reasoning', content=[Content(text='The user wants a short role-play (max 5 lines) of manager and employee discussing performance issues. The manager should talk like a pirate (from developer instruction). So manager must have pirate speech. The employee presumably normal speech.\n\nWe need no disallowed content. It\'s okay. Provide 5 lines max. Probably 4 lines: Manager, Employee alternating. Make sure manager uses pirate lingo: "Arr", "matey", etc.\n\nWe must comply with developer instruction: manager should talk like a pirate. Provide short role-play. 5 lines max. Let\'s output 4 lines.', type='reasoning_text')], encrypted_content=None, status='completed', format='unknown'),
 ResponseOutputMessage(id='msg_tmp_d3c2lt6gya7', content=[ResponseOutputText(annotations=[], text="**Manager (pirate):** Arr, ye scall

In [19]:
multi_turn_conversation.extend(first_turn_response.output)
pprint(multi_turn_conversation)

[{'content': 'Generate a short role-play in no more than 5 lines: manager and '
             'employee discussing performance issues.',
  'role': 'user'},
 ResponseReasoningItem(id='rs_tmp_l1dr2vg3lzk', summary=[], type='reasoning', content=[Content(text='The user wants a short role-play (max 5 lines) of manager and employee discussing performance issues. The manager should talk like a pirate (from developer instruction). So manager must have pirate speech. The employee presumably normal speech.\n\nWe need no disallowed content. It\'s okay. Provide 5 lines max. Probably 4 lines: Manager, Employee alternating. Make sure manager uses pirate lingo: "Arr", "matey", etc.\n\nWe must comply with developer instruction: manager should talk like a pirate. Provide short role-play. 5 lines max. Let\'s output 4 lines.', type='reasoning_text')], encrypted_content=None, status='completed', format='unknown'),
 ResponseOutputMessage(id='msg_tmp_d3c2lt6gya7', content=[ResponseOutputText(annotations=[], 

In [20]:
multi_turn_conversation.append({ "role": "user", "content": "Keep the role-play going for another 5 lines, but introduce a new development: the employee's wife enters the room dramatically and tells him that she's pregnant, leaving him genuinely surpirsed but at the same time desperately hoping to save his job because his family depends on it." })
second_turn_response = client.responses.create(
    model="openai/gpt-oss-120b",
    input=multi_turn_conversation,
    extra_body={
        "provider": { "sort": { "by": "throughput" } }
    }
)

In [21]:
print_response(second_turn_response)

Response id: gen-1783085691-1ZqVBEqkfMGWjm1h9Z35
Input tokens: 269 (0 cached); Output tokens: 284 (109 reasoning)
[ResponseReasoningItem(id='rs_tmp_6zw0g750qs', summary=[], type='reasoning', content=[Content(text="We need to continue role-play for another 5 lines. Must include new development: employee's wife enters dramatically and tells him she's pregnant. The employee is surprised but hopes to save job. Include manager (still pirate?) maybe keep previous tone. Ensure 5 lines, each a line of dialogue. Keep role-play format. Let's produce 5 lines: line1: wife enters; line2: employee reacts; line3: wife says; line4: employee pleads to manager; line5: manager responds. Ensure role-play continues.", type='reasoning_text')], encrypted_content=None, status='completed', format='unknown'),
 ResponseOutputMessage(id='msg_tmp_utq6ovubiha', content=[ResponseOutputText(annotations=[], text='**Wife (bursting in):** “Honey, I have news— I’m pregnant!”\n\n**Employee (stunned):** “What? … That… that

# Structured output

In [22]:
no_structure_response = client.responses.create(
    model="nvidia/nemotron-nano-12b-v2-vl:free",
    input="Generate a short role-play in no more than 5 lines: manager and employee discussing performance issues.",
)


In [23]:
print_response(no_structure_response)

Response id: gen-1783085885-Nrx7JLhvVDgIQXk3ZeFZ
Input tokens: 34 (0 cached); Output tokens: 204 (0 reasoning)
[ResponseOutputMessage(id='msg_tmp_ognkqgjyf1l', content=[ResponseOutputText(annotations=[], text="Manager: I've noticed some recent performance issues with the projects you've been handling. Can you help me understand what challenges you've been facing?\n\nEmployee: I appreciate your concern, but I believe there might be some misunderstandings. I've been dealing with unforeseen technical difficulties and some team members have been unavailable due to illness.\n\nManager: I see. It's important to communicate these challenges promptly. How do you plan to address these issues moving forward?\n\nEmployee: I understand the need for better communication. I'll ensure to provide regular updates on project progress and any obstacles we encounter. Additionally, I'll work on finding solutions for the technical problems and propose a plan to redistribute tasks to account for the team mem

`Structured output` evolves from `JSON mode`, however it is no longer recommended to use the latter for a number of reasons.

> **OpenAI:**
> _When using JSON mode, you must always instruct the model to produce JSON via some message in the conversation, for example via your system message. If you don't include an explicit instruction to generate JSON, the model may generate an unending stream of whitespace and the request may run continually until it reaches the token limit._

> **OpenAI:**
> _JSON mode will not guarantee the output matches any specific schema, only that it is valid and parses without errors._

In [24]:
diy_structured_response = client.responses.create(
    model="nvidia/nemotron-nano-12b-v2-vl:free",
    input="Generate a short role-play in no more than 5 lines: manager and employee discussing performance issues.",
    text={
        "format": {
            "type": "json_schema",
            "name": "character",
            "schema": {
              "type": "object",
              "properties": {
                  "name": {"type": "string", "description": "The character's full name"},
                  "age": {"type": "integer", "description": "The character's age"},
                  "hobby": {"type": "string", "description": "The character's hobby"}
              },
              "required": ["name", "age", "hobby"],
              "additionalProperties": False,
            },
            "strict": True
        }
    }
)

In [25]:
print_response(diy_structured_response)

Response id: gen-1783085894-dMsfNEnHZzs7yJTZf8ba
Input tokens: 34 (0 cached); Output tokens: 73 (0 reasoning)
[ResponseOutputMessage(id='msg_tmp_qmd9qat9fkm', content=[ResponseOutputText(annotations=[], text='{  \n  "name": "Practice 2023-12-04 15.15.24",  \n  "age": 1  \n  ,  \n  "hobby":  "engage in activities that you enjoy or do with passion and dedication."  \n  }', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)]

-------------------- [Text] --------------------
{  
  "name": "Practice 2023-12-04 15.15.24",  
  "age": 1  
  ,  
  "hobby":  "engage in activities that you enjoy or do with passion and dedication."  
  }


In [35]:
class Character(BaseModel):
    name: str
    age: int
    hobby: str

auto_structured_response = client.responses.parse(
    #model="nvidia/nemotron-nano-12b-v2-vl:free", - It was imposible for this model to generate structured output
    model="openai/gpt-oss-120b",
    input="Generate a short role-play in no more than 5 lines: manager and employee discussing performance issues.",
    text_format=Character
)

In [36]:
print_response(auto_structured_response)

print()
print(f"Parsed output: {auto_structured_response.output_parsed}")

Response id: gen-1783086646-89dG2xDfd2q0gF4khPiO
Input tokens: 74 (0 cached); Output tokens: 92 (61 reasoning)
[ResponseReasoningItem(id='rs_tmp_7cg0etvrbsk', summary=[], type='reasoning', content=[Content(text="We need to produce a short role-play, max 5 lines, with manager and employee discussing performance issues. Likely each line is a dialogue line. So perhaps 5 lines total, alternating between manager and employee. Keep concise. Let's produce.", type='reasoning_text')], encrypted_content=None, status='completed', format='unknown'),
 ParsedResponseOutputMessage[TypeVar](id='msg_tmp_pplv5tr2xc', content=[ParsedResponseOutputText[TypeVar](annotations=[], text='{\n  "name": "Performance Review Role‑Play",\n  "age": 2,\n  "hobby": "corporate storytelling"\n}', type='output_text', logprobs=[], parsed=Character(name='Performance Review Role‑Play', age=2, hobby='corporate storytelling'))], role='assistant', status='completed', type='message', phase=None)]

-------------------- [Text] ---

# Vision

In [37]:
image_analysis_response = client.responses.create(
    model="nvidia/nemotron-nano-12b-v2-vl:free",
    input=[
        {
            "role": "system",
            "content": "You are an expert image analyst. Keep your answer concise and structured."
        },
        {
            "role": "user",
            "content": [
                { "type": "input_text", "text": "Analyze this image and return a one-sentence summary followed by 5 key visible objects." },
                { "type": "input_image", "image_url": "https://freerangestock.com/sample/88947/painter-working-in-studio.jpg" }
            ]
        }
    ]
)

In [38]:
print_response(image_analysis_response)

Response id: gen-1783086719-NgcJbti0KwWCvacyOBXJ
Input tokens: 819 (0 cached); Output tokens: 68 (0 reasoning)
[ResponseOutputMessage(id='msg_tmp_m5epweb0wkm', content=[ResponseOutputText(annotations=[], text='In the room, a painter immersed in a creation sits before a canvas, surrounded by other creations. \n\n5 key visible objects:\n\n1.  Potrait being painted.\n2.  Table.\n3.  Paints.\n4.  Paintings in the background.\n5.  A womon painting a potrait.\n', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)]

-------------------- [Text] --------------------
In the room, a painter immersed in a creation sits before a canvas, surrounded by other creations. 

5 key visible objects:

1.  Potrait being painted.
2.  Table.
3.  Paints.
4.  Paintings in the background.
5.  A womon painting a potrait.

